# Step 12 — Unbiased validation

**Input:** `data/validation/unbiased_sample_v1.xlsx` (filled in) + `unbiased_sample_v1_key.parquet`
**Output:** `data/validation/12_unbiased_comparison.csv`

The first benchmark cannot rank classifiers. Its workbook arrived pre-filled with an
earlier LLM run's answers, so on the 719 "green" Bosserhof rows the new LLM run repeats
that answer 68.4% of the time and scores correct 68.4% of the time — the *same number*,
because on a green row the truth **is** the earlier answer. Repeat it → 100% correct;
deviate → 0%. On the 155 "red" rows the human overrode it, so repeating is automatically
wrong, and the run repeats it 52.9% of the time.

492 of the LLM's 500 correct Bosserhof answers sit in the green subset. Green rewards
agreeing with the seed and red punishes it. Neither measures classification.

This notebook scores both arms against labels written **from the raw evidence, with no
classifier output visible**, and separately measures how far the old workbook's "truth"
was shaped by what its reviewer was shown.

## Two things it reports

1. **A clean ranking.** Rule engine vs LLM on truth neither seeded. Because both arms
   label the *same* buildings, the paired McNemar test is used rather than comparing two
   independent proportions — at n=200 that is the difference between a usable answer and
   an inconclusive one.
2. **The acceptance bias**, measured directly: unbiased label vs workbook label on the same
   buildings. Where they disagree, the workbook was recording what the reviewer was
   willing to accept rather than what is there.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import (
    VALIDATION_DIR, ZONE_ACTIVITY_COLUMNS, BOSSERHOF_WEIGHTS,
    VALIDATION_ACTIVITY_TYPO_FIXES,
)
from validation_utils import (
    score_activities, score_bosserhof, label_accounting, bosserhof_accounting,
    collapse_to_zone_activities, clean_bosserhof, classes_mentioned,
    resolve_prediction_bosserhof, wilson_interval, BOSSERHOF_KNOWN,
)

import re
import pandas as pd

pd.set_option('display.width', 200)

UNBIASED_XLSX = VALIDATION_DIR / 'unbiased_sample_v1.xlsx'
UNBIASED_KEY  = VALIDATION_DIR / 'unbiased_sample_v1_key.parquet'
OUT        = VALIDATION_DIR / '12_unbiased_comparison.csv'

for p, what in [(UNBIASED_XLSX, 'the annotation workbook'), (UNBIASED_KEY, 'the withheld key')]:
    if not p.exists():
        raise FileNotFoundError(f'{p}\n  missing: {what}\n'
                                '  Generate with: python scripts/make_unbiased_sample.py')

sheet = pd.read_excel(UNBIASED_XLSX, sheet_name='annotate')
key   = pd.read_parquet(UNBIASED_KEY)
print(f'{len(sheet)} rows in the sheet, {len(key)} in the key')
assert set(sheet['row_id']) == set(key['row_id']), 'sheet and key do not correspond'
assert sheet['row_id'].is_unique, 'row_id repeated — rows were added or reordered'

filled = int(sheet['activities'].notna().sum())
print(f'activities answered : {filled}/{len(sheet)}')
print(f'bosserhof  answered : {int(sheet["bosserhof_class"].notna().sum())}/{len(sheet)}')
if filled == 0:
    raise SystemExit('The workbook is still empty — nothing to score yet.')

---
## Step 1 — Parse the answers

Same tolerance as notebook 09: activity names are word-scanned rather than parsed as a
list, because hand-typed cells arrive with broken brackets and quotes. `none` means "no
activity here", which is a real answer and not missing data.

A row is set aside if the annotator flagged `uncertain` or left the cell blank. Those are
excluded from **both** arms, so they cannot favour either.

In [ ]:
CANON = {}
for name in ZONE_ACTIVITY_COLUMNS:
    k = name.lower()
    CANON[k] = name
    CANON[k.replace('-', '_')] = name
    CANON[k.replace('_', '-')] = name
for typo, canonical in VALIDATION_ACTIVITY_TYPO_FIXES.items():
    CANON[typo.lower()] = canonical

NO_ACTIVITY = {'none', 'no', 'nothing', 'residential', 'living'}


def parse_activities(raw):
    # Returns (labels, ok). ok=False means unusable -> the row is excluded.
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None, False
    text = str(raw).strip()
    if text == '':
        return None, False
    if re.sub(r'[^a-z ]+', ' ', text.lower()).strip() in NO_ACTIVITY:
        return set(), True
    found = {CANON[t.lower()] for t in re.findall(r'[A-Za-z][A-Za-z_\-]*', text)
             if t.lower() in CANON}
    return (found, True) if found else (None, False)


def parse_bosserhof(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None, False
    v = resolve_prediction_bosserhof(raw)
    if v is None:
        return None, False
    if v == '' or v in BOSSERHOF_KNOWN:
        return v, True
    return None, False


uncertain = sheet['uncertain'].astype(str).str.strip().str.lower().isin(('x', 'yes', 'y', '1'))
print(f'flagged uncertain by the annotator: {int(uncertain.sum())}')

rows = []
unparsed_act, unparsed_boss = [], []
for _, r in sheet.iterrows():
    a, a_ok = parse_activities(r['activities'])
    b, b_ok = parse_bosserhof(r['bosserhof_class'])
    if uncertain.loc[_]:
        a_ok = b_ok = False
    if not a_ok and pd.notna(r['activities']) and str(r['activities']).strip():
        unparsed_act.append((r['row_id'], r['activities']))
    if not b_ok and pd.notna(r['bosserhof_class']) and str(r['bosserhof_class']).strip():
        unparsed_boss.append((r['row_id'], r['bosserhof_class']))
    rows.append({'row_id': r['row_id'],
                 'act_truth': a, 'act_ok': a_ok,
                 'boss_truth': b, 'boss_ok': b_ok})
blind = pd.DataFrame(rows)

# GUARD: a value the annotator typed but the parser could not read is a LOST label,
# not a neutral exclusion — it silently shrinks the sample. Name them rather than drop.
if unparsed_act or unparsed_boss:
    print('\nUNPARSEABLE ANSWERS — fix the spelling in the workbook, or extend the '
          'vocabulary, before trusting the numbers below:')
    for bid, v in unparsed_act[:10]:
        print(f'   activities {bid}: {v!r}')
    for bid, v in unparsed_boss[:10]:
        print(f'   bosserhof  {bid}: {v!r}')

print(f'\nscoreable — activities {int(blind.act_ok.sum())}, bosserhof {int(blind.boss_ok.sum())}')

---
## Step 2 — Score both arms on the unbiased truth

Same `validation_utils` functions as notebook 10, so these numbers sit on the same
footing as everything already computed.

In [ ]:
m = blind.merge(key, on='row_id', how='left', validate='one_to_one')
m['rule_zone'] = m['rule_mid_labels'].map(collapse_to_zone_activities)
m['llm_zone']  = m['llm_mid_labels'].map(collapse_to_zone_activities)

records = []
for arm, zone_col, boss_col in [('rule', 'rule_zone', 'rule_bosserhof'),
                                ('llm_blind', 'llm_zone', 'llm_bosserhof')]:
    a = m[m.act_ok]
    acc = label_accounting([(r.row_id, set(r[zone_col]), set(r.act_truth))
                            for _, r in a.iterrows()])
    records.append({'arm': arm, 'dimension': 'activities', **acc})

    b = m[m.boss_ok]
    bac = bosserhof_accounting([(r.row_id,
                                 resolve_prediction_bosserhof(r[boss_col]),
                                 r.boss_truth) for _, r in b.iterrows()])
    records.append({'arm': arm, 'dimension': 'bosserhof', 'buildings': bac['buildings'],
                    'correct': bac['correct'], 'accuracy': bac['accuracy'],
                    'accuracy_lo': bac['accuracy_lo'], 'accuracy_hi': bac['accuracy_hi']})

comp = pd.DataFrame(records)
comp.to_csv(OUT, index=False)
print(f'-> {OUT.name}\n')
print('ACTIVITIES')
print(comp[comp.dimension == 'activities'][
    ['arm', 'buildings', 'precision', 'precision_lo', 'precision_hi',
     'recall', 'recall_lo', 'recall_hi', 'exact_set_match']].round(3).to_string(index=False))
print('\nBOSSERHOF')
print(comp[comp.dimension == 'bosserhof'][
    ['arm', 'buildings', 'correct', 'accuracy', 'accuracy_lo', 'accuracy_hi']
].round(3).to_string(index=False))

---
## Step 3 — The paired test

Both arms labelled the *same* buildings, so comparing two independent proportions throws
away most of the information and at n=200 would call almost anything inconclusive.
McNemar uses only the **discordant** rows — where one arm is right and the other wrong —
which is where the evidence about a difference actually lives.

In [ ]:
def mcnemar(a_correct, b_correct, a_name, b_name):
    # Paired test on the discordant rows only — that is where the evidence about
    # a difference lives. Comparing two independent proportions at n=200 throws
    # most of it away.
    b01 = int((~a_correct & b_correct).sum())   # only B right
    b10 = int((a_correct & ~b_correct).sum())   # only A right
    n = b01 + b10
    print(f'  both right      : {int((a_correct & b_correct).sum())}')
    print(f'  both wrong      : {int((~a_correct & ~b_correct).sum())}')
    print(f'  only {a_name:9s}  : {b10}')
    print(f'  only {b_name:9s}  : {b01}')
    if n == 0:
        print('  the arms never disagree — no evidence either way'); return
    # Exact binomial two-sided p on the discordant pairs.
    from math import comb
    k = min(b01, b10)
    p = min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)
    lo, hi = wilson_interval(b10, n)
    print(f'  discordant      : {n}')
    print(f'  P(only {a_name} right | disagree) = {b10/n:.1%}  95% CI [{lo:.1%}, {hi:.1%}]')
    print(f'  exact McNemar p = {p:.4f}   ->  ' +
          ('a real difference' if p < 0.05 else
           'NOT distinguishable at this sample size'))


b = m[m.boss_ok].copy()
b['rule_ok'] = b.apply(lambda r: (resolve_prediction_bosserhof(r.rule_bosserhof) or '')
                       == (r.boss_truth or ''), axis=1)
b['llm_ok'] = b.apply(lambda r: (resolve_prediction_bosserhof(r.llm_bosserhof) or '')
                      == (r.boss_truth or ''), axis=1)
print(f'BOSSERHOF — paired, n={len(b)}')
mcnemar(b.rule_ok, b.llm_ok, 'rule', 'llm')

a = m[m.act_ok].copy()
a['rule_ok'] = a.apply(lambda r: set(r.rule_zone) == set(r.act_truth), axis=1)
a['llm_ok'] = a.apply(lambda r: set(r.llm_zone) == set(r.act_truth), axis=1)
print(f'\nACTIVITIES (exact set match) — paired, n={len(a)}')
mcnemar(a.rule_ok, a.llm_ok, 'rule', 'llm')

---
## Step 4 — How biased was the original workbook?

The same buildings carry a workbook label and a unbiased label. Where they disagree, the
workbook was recording what its reviewer was willing to **accept** rather than what the
evidence supports — and because that reviewer was shown an LLM's answers, the
disagreements should lean toward the LLM.

The decisive number is the last one: of the rows where unbiased and workbook disagree, how
often does the workbook side with the LLM? Materially above 50% means the old benchmark
was tilted, by roughly that margin.

In [ ]:
w = m[m.boss_ok & m.workbook_bosserhof.notna()].copy()
w['wb'] = w.workbook_bosserhof.fillna('')
w['agree'] = w.wb.str.lower() == w.boss_truth.fillna('').str.lower()
print(f'buildings with both a workbook and a unbiased label: {len(w)}')
print(f'  they agree: {int(w.agree.sum())} ({w.agree.mean():.1%})')

d = w[~w.agree].copy()
if len(d):
    d['wb_is_llm'] = d.apply(
        lambda r: (resolve_prediction_bosserhof(r.llm_bosserhof) or '').lower()
        == r.wb.lower(), axis=1)
    d['wb_is_rule'] = d.apply(
        lambda r: (resolve_prediction_bosserhof(r.rule_bosserhof) or '').lower()
        == r.wb.lower(), axis=1)
    lo, hi = wilson_interval(int(d.wb_is_llm.sum()), len(d))
    print(f'\nwhere they DISAGREE (n={len(d)}), the workbook label matches:')
    print(f'  the LLM  : {int(d.wb_is_llm.sum())} ({d.wb_is_llm.mean():.1%}) '
          f'95% CI [{lo:.1%}, {hi:.1%}]')
    print(f'  the rules: {int(d.wb_is_rule.sum())} ({d.wb_is_rule.mean():.1%})')
    print('\n  A figure well above 50% is the acceptance bias, quantified: the old')
    print('  "truth" tracked the answers its reviewer was shown.')

print('\nBosserhof accuracy, unbiased truth vs workbook truth, same buildings:')
for arm, col in [('rule', 'rule_bosserhof'), ('llm_blind', 'llm_bosserhof')]:
    vs_blind = w.apply(lambda r: (resolve_prediction_bosserhof(r[col]) or '')
                       == (r.boss_truth or ''), axis=1).mean()
    vs_wb = w.apply(lambda r: (resolve_prediction_bosserhof(r[col]) or '').lower()
                    == r.wb.lower(), axis=1).mean()
    print(f'  {arm:10s} vs unbiased {vs_blind:6.1%}   vs workbook {vs_wb:6.1%}   '
          f'delta {vs_wb - vs_blind:+.1f} pp')

---
## Done

`12_unbiased_comparison.csv` holds the only classifier ranking in this repo that neither
classifier seeded.

Reading it:

- **Use the McNemar result, not the two accuracies**, to say whether one arm is better.
  At n=200 the individual intervals overlap for differences the paired test resolves
  comfortably.
- The `delta` in Step 4 is what the original benchmark was worth: how many points each
  arm gains from being scored against reviewer-accepted labels instead of blind ones.
- Rows the annotator flagged uncertain or left blank are excluded from **both** arms, so
  exclusions cannot favour either.